# Proyecto - Aprendizaje de Máquina

## Librerías

In [10]:
import os
import pandas as pd
import re
import matplotlib.pyplot as plt
import nltk
nltk.download('punkt_tab')
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('stopwords')
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Brixt\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Brixt\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Limpieza y transformación de datos

### Carga de datos

In [11]:
# Carga el archivo CSV
filepath = os.path.join(os.getcwd(), 'data/train.csv')
with open(filepath, 'r', encoding='utf-8') as f:
    # Guarda el header
    header = f.readline()
    # raw_data almacena los pares de texto y década
    raw_data = [header.strip().split(',')]
    content = f.read() # Archivo con datos completos
    # Extrae los datos a partir de una expresión regular que busca 
    # texto entre comillas dobles seguido de una coma, tres dígitos y un salto de línea
    regular_expression = r'"(?:[^"]|"")*",\d{3}\n'
    joint_data = re.findall(regular_expression, content, flags=re.MULTILINE) #Se guarda en una lista
    for data in joint_data:
        data = data.strip().replace('\n', '') # Elimina saltos de línea y espacios extra
        text = data[:-4] # Texto sin la década ni la coma separadora
        text = text[1:-1].strip() # Elimina comillas externas y espacios extra
        decade = data[-3:] # Década (últimos 3 caracteres)
        raw_data.append([text, int(decade)])
print("Total de registros cargados:", len(raw_data) - 1) # Resta 1 por el header


Total de registros cargados: 31380


In [12]:
data = pd.DataFrame(raw_data[1:], columns=raw_data[0])
print(data.sample(5))

                                                    text  decade
18128  defde las diez de la manana del día prim^ repr...     176
25078  Cuando  el  Rey,  se  ausenta  de  la  ciudad ...     185
2251   fe  h¡z¡eron,y lugares adonde.y  fus  dcfcrip-...     162
208    etftípéerken  Griego  fe  lhma  aluraeri  ¿  q...     157
10981  t Elfegiinboesbéló^esfigurabOpoS TOibíéafuta p...     155


In [13]:
print("Registros únicos:", data["text"].nunique())
duplicate_rows = data[data.duplicated(subset=['text'], keep=False)]
print(duplicate_rows.sort_values(by='text'))

Registros únicos: 31335
                                                    text  decade
28956  (ft.Ciem.i.ae crlrb. fnil%rer>>xi,Cóc. Ba (il....     156
20852  (ft.Ciem.i.ae crlrb. fnil%rer>>xi,Cóc. Ba (il....     156
17691  + Añiado;que fi en laforima- cion del bezerro,...     166
5779   + Añiado;que fi en laforima- cion del bezerro,...     166
6828   + Haga un uso exclusivamente no comercial de e...     181
...                                                  ...     ...
23899  s]2lltaniccloquéda.ii\a.Dcon-o íftruméto cardc...     151
20331  tentia Paul.de Caft.in.d.l.fi. C. commu.de leg...     151
12834  tentia Paul.de Caft.in.d.l.fi. C. commu.de leg...     151
20364  vel metu extortũ:vel in alterius detrimentuʒ: ...     153
5886   vel metu extortũ:vel in alterius detrimentuʒ: ...     153

[86 rows x 2 columns]


In [14]:
data.drop_duplicates(subset=['text'], inplace=True)
print("Total de registros después de eliminar duplicados:", len(data))

Total de registros después de eliminar duplicados: 31335


In [15]:
stemmer = PorterStemmer()

In [16]:
def clean_text(text):
    #Unir palabras con guiones
    text = re.sub(r'(\w+)-\s+(\w+)', r'\1\2', text)
    text = re.sub(r'[^a-zA-ZáéíóúüÁÉÍÓÚÜñÑ\s]', '', text) # Mantener solo caracteres alfabéticos y espacios
    tokens = word_tokenize(text)
    text = " ".join([t for t in tokens if t.isalnum()]) # Stemming
    return text.lower() # Convierte a minúsculas

In [17]:
clean_data = data.copy()
clean_data['text'] = clean_data['text'].apply(clean_text)
print(clean_data.sample(5))

                                                    text  decade
2668   cslaraña s f caíaracte oiscau agualigue catami...     180
29028  ai ve nsé que albricias me diera vueflra altez...     163
6413   ij no beuas de aqui adelante agua fi no vfá de...     156
19299  ioj yendo pites el dicho á cumplir fu promefe ...     175
22610  mos que chriftó nueítro señor el dia vltimo de...     167


In [18]:
stop_words_es = set(nltk.corpus.stopwords.words('spanish'))
vectorizer = TfidfVectorizer(stop_words=list(stop_words_es))
X_tfidf = vectorizer.fit_transform(clean_data['text'])
print("Vocabulario:", vectorizer.get_feature_names_out())

Vocabulario: ['aa' 'aaa' 'aaaaa' ... 'üñ' 'üü' 'üübcrio']
